<img src="../../branding/aa-wordmark.png" alt="AA Engineering Toolkit" width="320px">

*Areeb Ahmed | Mechanical Engineer*

# Battery modelling: equivalent circuit, thermal and aging

This notebook walks through the browser-ready battery models in `engtoolkit.electrical.battery`
(pure NumPy, the same code the web app runs):

1. **Cells** from `engtoolkit.catalogs` (capacity, OCV-SOC table, 2RC parameters, thermal properties)
2. **Equivalent-circuit model (ECM)**: Thevenin 1RC/2RC with OCV(SOC) and Arrhenius resistances
3. **Thermal model**: lumped `m·cp·dT/dt = Q_gen - h·A·(T - T_coolant)` with `h` from `engtoolkit.thermo.heat_transfer`
   and coolant properties from `engtoolkit.materials.fluid`
4. **Aging**: semi-empirical capacity fade and resistance growth (Wang et al. 2011 for LFP, Schmalstieg et al. 2014 for NMC)
5. **PyBaMM reference**: the precomputed physics-based results the web app overlays

Units are SI (A, V, W, J, K, s) except capacity in A·h.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from engtoolkit import run
from engtoolkit.catalogs import get_cell, list_cells
from engtoolkit.core.plotting import use_style
from engtoolkit.electrical.battery import ECMParams, aging, simulate

use_style()
pd.DataFrame([{
    "cell": c.name, "chemistry": c.chemistry, "capacity [Ah]": c.capacity_Ah, "V nominal": c.V_nominal,
    "mass [g]": c.mass * 1e3, "Wh/kg": round(c.specific_energy), "R0 [mΩ]": c.R0 * 1e3, "aging model": c.aging_model,
} for c in map(get_cell, list_cells())])

## 1. Open-circuit voltage

SOC 0 is where the OCV reaches the discharge cut-off. LFP is famously flat in the middle, which makes SOC
estimation from voltage hard; NMC has a usable slope.

In [ ]:
soc = np.linspace(0, 1, 201)
fig, ax = plt.subplots(figsize=(8, 4.5))
for name in list_cells():
    ax.plot(soc * 100, get_cell(name).ocv_at(soc), label=name)
ax.set(xlabel="SOC [%]", ylabel="OCV [V]", title="Open-circuit voltage")
ax.legend();

## 2. Equivalent-circuit model: step response

With current $I$ held constant, each RC pair relaxes exponentially, so the terminal voltage is

$$V(t) = OCV - I R_0 - I R_1 (1 - e^{-t/\tau_1}) - I R_2 (1 - e^{-t/\tau_2})$$

The simulator uses this exact solution step by step, so it reproduces the curve for any time step
(checked in `tests/electrical/test_battery.py`). For the LG M50 the 2RC parameters were fitted to PyBaMM
DFN pulses: the fast pair (τ₁ ≈ 11 s) is charge transfer, the slow pair (τ₂ ≈ 86 s) diffusion.

In [ ]:
cell = get_cell("LG M50 21700 (NMC811)")
p = ECMParams.from_cell(cell)
sim = simulate(p, current_fn=lambda t: 5.0 if t < 300 else 0.0, soc0=0.6, dt=1.0, t_max=900)
t = sim.t[sim.t < 300]
analytical = cell.ocv_at(0.6 - t / 3600) - 5 * (p.R0 + p.R1 * (1 - np.exp(-t / p.tau1)) + p.R2 * (1 - np.exp(-t / p.tau2)))
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(sim.t, sim.voltage, label="2RC model")
ax.plot(t, analytical, "--", label="analytical (pulse)")
ax.set(xlabel="Time [s]", ylabel="Voltage [V]", title="1C pulse and relaxation at 60 % SOC")
ax.legend();

## 3. Discharge with thermal coupling

Heat generation is the Joule and polarisation loss plus the reversible (entropic) heat
(Bernardi et al. 1985): `Q = I²R0 + V1²/R1 + V2²/R2 - I·T·dU/dT`. Resistances fall with temperature,
so a warmer cell sags less. The battery simulator tool returns chart specs that draw with matplotlib.

In [ ]:
res = run("electrical.battery_simulator", c_rates="0.5, 1, 2", cooling="natural_air", aging_cycles=0)
pd.DataFrame(res.rows, columns=res.columns)

In [ ]:
for chart in res.chart_specs[:2]:
    chart.to_figure()

## 4. Cooling: air, forced air or a water-glycol cold plate

- still air: Churchill-Chu natural convection on a horizontal cylinder + linearised radiation
- forced air: Churchill-Bernstein for a cylinder in cross-flow (flat plate for pouch cells)
- cold plate: 30 % ethylene glycol (`INCOMP::MEG[0.3]`) in a 6 mm channel, Gnielinski / laminar Nu,
  in series with a 1 mm gap pad and a 2 mm aluminium wall over 25 % of the cell surface

In [ ]:
cmp = run("electrical.cooling_comparison", c_rate=1.5)
print(f"h forced air = {cmp.h_forced:.1f} W/m²K, h channel = {cmp.h_channel:.0f} W/m²K (Re = {cmp.Re_channel:.0f})")
pd.DataFrame(cmp.rows, columns=cmp.columns)

In [ ]:
cmp.chart_specs[0].to_figure();

## 5. Aging

**LFP** (Wang et al. 2011): `Q_loss[%] = B(C)·exp(-(31700 - 370.3·C)/(R·T))·Ah^0.55`. A higher C-rate lowers the
activation energy, and temperature dominates.

**NMC** (Schmalstieg et al. 2014): calendar `α(V,T)·t^0.75` plus cycle `β(V̄, ΔDOD)·√Q` capacity loss, and
resistance growth `α_res·t^0.75 + β_res·Q`.

Below, the LFP cell cycled at different temperatures and C-rates. The cycle-life tool takes the cell
temperature from the thermal model automatically.

In [ ]:
cycles = np.linspace(0, 4000, 201)
fig, ax = plt.subplots(figsize=(8, 4.5))
for T in (288.15, 298.15, 318.15):
    for c in (0.5, 2, 6):
        curve = aging.cycle_aging("wang2011_lfp", cycles, c_rate=c, T=T)
        ax.plot(cycles, curve.capacity * 100, label=f"{T - 273.15:.0f} °C, {c:g}C")
ax.axhline(80, color="grey", lw=0.8)
ax.set(xlabel="Full-equivalent cycles", ylabel="Capacity [%]", title="LFP cycle aging (Wang et al. 2011)")
ax.legend(ncol=3, fontsize=8);

In [ ]:
life = run("electrical.cycle_life", c_rates="0.5, 1, 2", cooling="natural_air", n_cycles=2000)
print(life.model)
pd.DataFrame(life.rows, columns=life.columns)

In [ ]:
for chart in life.chart_specs:
    chart.to_figure()

## 6. PyBaMM reference

`scripts/generate_battery_reference.py` ran PyBaMM (DFN with lumped thermal; SPM with SEI growth and lithium
plating) for the LG M50 and saved the results in `electrical/data/pybamm_reference.json`. With the same heat
transfer coefficient the ECM reproduces the DFN discharge capacity within a few percent and the temperature
within a few kelvin. See `02_pybamm.ipynb` to run PyBaMM yourself.

In [ ]:
ref = run("electrical.pybamm_reference", T_ambient=298.15)
print(f"max ECM vs DFN capacity difference: {ref.max_capacity_error:.1%}")
pd.DataFrame(ref.rows, columns=ref.columns)

In [ ]:
for chart in ref.chart_specs[:3]:
    chart.to_figure()